# 🔥 World Fire Propagation Map - Demo

This notebook demonstrates the fire spread simulation and optimization capabilities.

## Setup

In [ ]:
# Import required modules
import sys
sys.path.insert(0, '.')

from modules.simulation import FireSpreadSimulator, SimulationConfig, run_parameter_sweep
from modules.export import DataExporter
import matplotlib.pyplot as plt
import numpy as np
import json

## 1. Basic Fire Spread Simulation

In [ ]:
# Create and run a simulation
config = SimulationConfig(
    grid_size=7,
    lambda_spread=0.005,
    num_firefighters=1,
    fire_start_nodes=[24],  # Center of 7x7 grid
    seed=42
)

simulator = FireSpreadSimulator(config)

# Run with greedy strategy
result = simulator.run(firefighter_strategy="greedy")

print("=" * 40)
print("FIRE SPREAD SIMULATION RESULTS")
print("=" * 40)
print(f"Grid Size: {config.grid_size}x{config.grid_size}")
print(f"Fire Spread Rate (λ): {config.lambda_spread}")
print(f"Firefighters: {config.num_firefighters}")
print()
print(f"🔥 Burned Nodes: {result.total_burned}")
print(f"🛡️  Protected Nodes: {result.total_protected}")
print(f"⏱️  Time Steps: {result.time_steps}")
print(f"📍 Firefighter Placements: {result.firefighter_placements}")

## 2. Strategy Comparison

In [ ]:
# Compare different firefighter placement strategies
simulator.reset()
comparison = simulator.compare_strategies()

print("STRATEGY COMPARISON")
print("-" * 40)
print(f"{'Strategy':<12} {'Burned':<10} {'Protected':<12} {'Time Steps':<12}")
print("-" * 40)

for strategy, res in comparison.items():
    print(f"{strategy.capitalize():<12} {res.total_burned:<10} {res.total_protected:<12} {res.time_steps:<12}")

print("-" * 40)
print("\n🔥 Greedy strategy protects highest-degree nodes first")
print("🎲 Random strategy places firefighters randomly")
print("📍 Central strategy protects center of fire spread")

## 3. Grid Visualization

In [ ]:
# Visualize the final state
def visualize_grid(grid, ax):
    """Create a colored visualization of the grid."""
    colors = {
        0: '#f0f0f0',  # Unburned (white)
        1: '#ff4444',  # Burning (red)
        2: '#44ff44',  # Protected (green)
        3: '#333333'   # Burned (dark gray)
    }
    
    color_grid = np.array([[colors.get(v, '#cccccc') for v in row] for row in grid])
    ax.imshow(color_grid, interpolation='nearest')
    
    # Add grid lines
    for i in range(grid.shape[0] + 1):
        ax.axhline(i - 0.5, color='black', linewidth=0.5)
        ax.axvline(i - 0.5, color='black', linewidth=0.5)
    
    ax.set_xticks(range(grid.shape[1]))
    ax.set_yticks(range(grid.shape[0]))
    ax.set_xticklabels([])
    ax.set_yticklabels([])

# Create visualization
fig, ax = plt.subplots(figsize=(8, 8))
grid = simulator.get_grid_visualization()
visualize_grid(grid, ax)

ax.set_title('Fire Spread Result\n(Red=Burning, Green=Protected, Dark=Burned, White=Unburned)')
plt.tight_layout()
plt.show()

## 4. Parameter Sensitivity Analysis

In [ ]:
# Run parameter sweep
print("Running parameter sweep...")
print("This may take a moment...\n")

results = run_parameter_sweep(
    grid_size=7,
    lambda_values=[0.001, 0.005, 0.01, 0.02],
    firefighter_values=[1, 2, 3]
)

# Display results
print("PARAMETER SENSITIVITY RESULTS")
print("=" * 60)

for cfg in results["configurations"]:
    print(f"\nλ={cfg['lambda']}, Firefighters={cfg['firefighters']}")
    for strategy, res in cfg["results"].items():
        print(f"  {strategy:8s}: {res['burned']:2d} burned, {res['protected']:2d} protected")

## 5. Data Export

In [ ]:
# Export simulation results
from modules.export import export_simulation_result

simulation_result = {
    "total_burned": result.total_burned,
    "total_protected": result.total_protected,
    "time_steps": result.time_steps,
    "burned_nodes": result.burned_nodes,
    "protected_nodes": result.protected_nodes,
    "firefighter_placements": result.firefighter_placements
}

simulation_config = {
    "grid_size": config.grid_size,
    "lambda_spread": config.lambda_spread,
    "num_firefighters": config.num_firefighters,
    "strategy": "greedy"
}

# Export as JSON
json_output = export_simulation_result(simulation_result, simulation_config, "json")

# Export as Markdown report
md_output = export_simulation_result(simulation_result, simulation_config, "markdown")

print("Exported Results:")
print("-" * 40)
print(json_output[:500])
print("...\n")
print("Markdown Report:")
print("-" * 40)
print(md_output)

## 6. Fire Progression Animation

In [ ]:
# Animate fire progression
from matplotlib.animation import FuncAnimation

fig, ax = plt.subplots(figsize=(8, 8))

def update_frame(frame):
    ax.clear()
    
    if frame < len(result.fire_progression):
        fire_set = result.fire_progression[frame]
        
        # Create grid for this frame
        grid = np.zeros((7, 7), dtype=int)
        for i in range(49):
            if i in fire_set:
                grid[i // 7, i % 7] = 1
        
        visualize_grid(grid, ax)
        ax.set_title(f'Fire Spread - Time Step {frame}')
    else:
        ax.text(0.5, 0.5, 'Simulation Complete', 
                ha='center', va='center', fontsize=20)
        ax.set_title('Final State')

ani = FuncAnimation(fig, update_frame, frames=len(result.fire_progression) + 5, 
                    interval=500, repeat=False)

plt.tight_layout()
plt.show()

print("\n🔥 Animation shows fire spread over time")
print("🛡️  Green cells show firefighter-protected areas")

## Summary

This demo shows:
1. ✅ Fire spread simulation on grid graphs
2. ✅ Firefighter placement optimization
3. ✅ Strategy comparison (greedy, random, central)
4. ✅ Parameter sensitivity analysis
5. ✅ Data export (JSON, Markdown)
6. ✅ Visualization and animation

For the full dashboard with real NASA FIRMS data, run:
```bash
python run_app.py
```